## RAG Pipelines : Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_3264\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\Lenovo\Desktop\AgenticAI_Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: anish_daha_data_science_resume.pdf
  ✓ Loaded 1 pages

Processing: nec-syllabus.pdf
  ✓ Loaded 6 pages

Total documents loaded: 7


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-07T09:14:20+00:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-07T09:14:20+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '..\\data\\pdf\\anish_daha_data_science_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anish_daha_data_science_resume.pdf', 'file_type': 'pdf'}, page_content='Anish Dahal\nAI Data Scientist\n+977-9810564290 | aneeshdahal5@gmail.com | linkedin.com/in/aneesh-dahal-98b171338 |\ngithub.com/anees555\nProfessional Summary\nComputer Engineering graduate specializing in AI and Data Science, with hands-on experience building machine learning\npipelines, retrieval-augmented generation (RAG) systems, and business intelligence solutions using Python, SQL, and Power\nBI. Skilled in NLP, vector search, 

#### Text Splitting get into Chunks

In [4]:
def split_documents(documents,chunk_size=500,chunk_overlap=100):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 7 documents into 65 chunks

Example chunk:
Content: Anish Dahal
AI Data Scientist
+977-9810564290 | aneeshdahal5@gmail.com | linkedin.com/in/aneesh-dahal-98b171338 |
github.com/anees555
Professional Summary
Computer Engineering graduate specializing in...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-07T09:14:20+00:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-07T09:14:20+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '..\\data\\pdf\\anish_daha_data_science_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anish_daha_data_science_resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-07T09:14:20+00:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-07T09:14:20+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '..\\data\\pdf\\anish_daha_data_science_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anish_daha_data_science_resume.pdf', 'file_type': 'pdf'}, page_content='Anish Dahal\nAI Data Scientist\n+977-9810564290 | aneeshdahal5@gmail.com | linkedin.com/in/aneesh-dahal-98b171338 |\ngithub.com/anees555\nProfessional Summary\nComputer Engineering graduate specializing in AI and Data Science, with hands-on experience building machine learning\npipelines, retrieval-augmented generation (RAG) systems, and business intelligence solutions using Python, SQL, and Power'),
 Document(metadata={'producer': '

### Embedding and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\Lenovo\Desktop\AgenticAI_Langchain\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Lenovo\Desktop\AgenticAI_Langchain\.venv\Lib\site-packages\huggingface

Model loaded successfully. Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_3264\3958577153.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [9]:
vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-07T09:14:20+00:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-07T09:14:20+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '..\\data\\pdf\\anish_daha_data_science_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anish_daha_data_science_resume.pdf', 'file_type': 'pdf'}, page_content='Anish Dahal\nAI Data Scientist\n+977-9810564290 | aneeshdahal5@gmail.com | linkedin.com/in/aneesh-dahal-98b171338 |\ngithub.com/anees555\nProfessional Summary\nComputer Engineering graduate specializing in AI and Data Science, with hands-on experience building machine learning\npipelines, retrieval-augmented generation (RAG) systems, and business intelligence solutions using Python, SQL, and Power'),
 Document(metadata={'producer': '

In [11]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

texts

['Anish Dahal\nAI Data Scientist\n+977-9810564290 | aneeshdahal5@gmail.com | linkedin.com/in/aneesh-dahal-98b171338 |\ngithub.com/anees555\nProfessional Summary\nComputer Engineering graduate specializing in AI and Data Science, with hands-on experience building machine learning\npipelines, retrieval-augmented generation (RAG) systems, and business intelligence solutions using Python, SQL, and Power',
 'BI. Skilled in NLP, vector search, ETL development, and dimensional data modeling across end-to-end AI and analytics\nprojects. Focused on transforming structured and unstructured data into actionable insights and intelligent applications.\nTechnical Skills\nProgramming: Python, SQL\nDatabases: PostgreSQL, Microsoft SQL Server\nBusiness Intelligence : Power BI, Power Query, DAX\nData Analysis : Pandas, NumPy, Matplotlib, Microsoft Excel',
 'Data Analysis : Pandas, NumPy, Matplotlib, Microsoft Excel\nData Modeling & ETL : Semantic Modeling, Star Schema, Dimensional Modeling, ETL, Medalli

In [12]:
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 65 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00,  3.47it/s]

Generated embeddings with shape: (65, 384)
Adding 65 documents to vector store...
Successfully added 65 documents to vector store
Total documents in collection: 65


### Retriever Pipeline From VectorStore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [14]:
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [15]:
rag_retriever

In [20]:
rag_retriever.retrieve("Tell me about the Electrical and Electronic syllabus of NEC ")

Retrieving documents for query: 'Tell me about the Electrical and Electronic syllabus of NEC '
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_c0ff5c4f_8',
  'content': '1 \nNepal Engineering Council Registration Examination \nComputer Engineering Syllabus (ACtE) \n \nChapters 1-2 are fundamentals/principles of concepts in computer engineering; chapters 3-9 are related to \napplication of computer engineering principles in practice; and the last (10 th) chapter is related to project \nplanning, design and implementation. \n \n1. Concept of Basic Electrical and Electronics Engineering     (AExE01)',
  'metadata': {'page': 0,
   'file_type': 'pdf',
   'content_length': 425,
   'producer': 'iLovePDF',
   'author': 'tenda',
   'doc_index': 8,
   'source_file': 'nec-syllabus.pdf',
   'title': 'Microsoft Word - Computer Engineering-final',
   'moddate': '2023-01-04T15:40:47+00:00',
   'page_label': '1',
   'creator': 'Microsoft® Word 2016',
   'source': '..\\data\\pdf\\nec-syllabus.pdf',
   'total_pages': 6,
   'creationdate': '2022-09-20T13:10:25+00:00'},
  'similarity_score': 0.1623392105102539,
  'distance': 0.83766

### RAG Pipeline- VectorDB To LLM Output Generation

In [23]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()



True

In [24]:
llm=ChatGroq(model="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

## Simple RAG Function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k = 3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"

    ## generate the answer using groq LLM
    prompt = f"""Use the following context to answer the question concisely.
            context:
            {context}

            Question: {query}

            Answer: """

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [25]:
answer = rag_simple("Who is Anish Dahal", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Who is Anish Dahal'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Anish Dahal is an AI and Data Scientist with a Computer Engineering background, experienced in building machine‑learning pipelines, retrieval‑augmented generation systems, and business‑intelligence solutions using Python, SQL, and Power platforms.


### Enhanced RAG pipeline Features

In [26]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [35]:
# Example usage:
result = rag_advanced("Electrical and Electronics in NEC syllabus", rag_retriever, llm, top_k=3, min_score=0.01, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Electrical and Electronics in NEC syllabus'
Top K: 3, Score threshold: 0.01
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: **Electrical & Electronics in the NEC Computer Engineering Syllabus**

- **Chapter 1 – “Concept of Basic Electrical and Electronics Engineering” (AExE01)**  
  * Covers the fundamentals of electrical and electronics engineering.  
  * Topics include basic electrical concepts, circuit theory, electronic components, signal processing fundamentals, and introductory power‑electronics principles.  
  * Provides the foundational knowledge required for applying electrical and electronics principles throughout the rest of the computer engineering curriculum.
Sources: [{'source': 'nec-syllabus.pdf', 'page': 0, 'score': 0.13488179445266724, 'preview': '1 \nNepal Engineering Council Registration Examination \nComputer Engineering Syllabus (ACtE) \n \nChapters 1-2 are fundamentals/principles of concepts in computer engineering; chapters 3-9 are related to \napplication of computer engineering principles in practice; and the last (10 th) chapter is relate...'}]
Confidence: 0.134881794452667